In [1]:
import os
import csv
import tkinter as tk
from PIL import Image, ImageTk

# Configuration
DATASET_ROOT = "./sample_dataset"
CSV_PATH = os.path.join(DATASET_ROOT, "metadata.csv")
IMAGE_FOLDERS = ["reference", "masks", "overlays"]

class DatasetViewer:
    def __init__(self, root):
        self.root = root
        self.root.title("CARLA Tri-View Dataset Viewer")
        
        # 1. Load metadata
        self.data = self.load_metadata()
        self.total_samples = len(self.data)
        self.current_index = 0
        
        if self.total_samples == 0:
            tk.Label(root, text="No data found in metadata.csv!").pack()
            return

        # 2. Setup UI Elements
        # A container frame to hold our three images side-by-side
        self.image_frame = tk.Frame(root)
        self.image_frame.pack(pady=10)
        
        self.image_labels = []
        self.photos = [None, None, None] # Crucial: prevents garbage collection of images
        
        # Create a column for each image type
        for folder in IMAGE_FOLDERS:
            container = tk.Frame(self.image_frame)
            container.pack(side=tk.LEFT, padx=10) # 10px padding between images
            
            title = tk.Label(container, text=folder.capitalize(), font=("Arial", 12, "bold"))
            title.pack()
            
            img_lbl = tk.Label(container)
            img_lbl.pack()
            
            self.image_labels.append(img_lbl)

        # Info text (Made it a single line so it takes up less vertical space)
        self.info_var = tk.StringVar()
        self.info_label = tk.Label(root, textvariable=self.info_var, font=("Courier", 12), justify=tk.CENTER)
        self.info_label.pack(pady=10)

        # Slider
        self.slider = tk.Scale(root, from_=0, to=self.total_samples - 1, orient=tk.HORIZONTAL, 
                               length=800, command=self.on_slider_move, showvalue=False)
        self.slider.pack(pady=10)

        # Instructions
        tk.Label(root, text="Use Left/Right Arrow Keys to navigate | Drag slider to jump").pack(pady=5)

        # 3. Bind Keys
        self.root.bind('<Right>', self.next_image)
        self.root.bind('<Left>', self.prev_image)

        # 4. Load the first batch of images
        self.update_view()

    def load_metadata(self):
        """Reads the CSV file so we know what to load."""
        data = []
        try:
            with open(CSV_PATH, mode='r') as f:
                reader = csv.DictReader(f)
                for row in reader:
                    data.append(row)
        except FileNotFoundError:
            print(f"Error: Could not find {CSV_PATH}")
        return data

    def update_view(self):
        """Dynamically loads all three images from the disk."""
        row = self.data[self.current_index]
        filename = f"{row['filename']}.png" 
        
        # Load Reference, Mask, and Overlay
        for i, folder in enumerate(IMAGE_FOLDERS):
            img_path = os.path.join(DATASET_ROOT, folder, filename)
            try:
                img = Image.open(img_path)
                # Resize slightly so all 3 fit nicely on standard monitors
                img = img.resize((400, 400), Image.Resampling.LANCZOS) 
                self.photos[i] = ImageTk.PhotoImage(img)
                self.image_labels[i].config(image=self.photos[i], text="")
            except Exception as e:
                # If an image didn't save properly during generation, show an error box
                self.photos[i] = None
                self.image_labels[i].config(image='', text=f"Missing:\n{folder}/{filename}", width=40, height=20, bg="lightgrey")
                print(f"Error loading {img_path}: {e}")

        # Update Text Info
        info_text = (
            f"File: {filename} ({self.current_index + 1}/{self.total_samples}) | Vehicle: {row['vehicle_id']}\n"
            f"SampleSet No: {row['sampleset_no']} | Location ID: {row['location_id']}\n"
            f"Camera: Dist={row['distance']}, Pitch={row['pitch']}, Yaw={row['yaw']}"
        )
        self.info_var.set(info_text)
        
        # Sync slider position
        self.slider.set(self.current_index)

    def next_image(self, event=None):
        if self.current_index < self.total_samples - 1:
            self.current_index += 1
            self.update_view()

    def prev_image(self, event=None):
        if self.current_index > 0:
            self.current_index -= 1
            self.update_view()

    def on_slider_move(self, value):
        self.current_index = int(value)
        self.update_view()

if __name__ == "__main__":
    root = tk.Tk()
    app = DatasetViewer(root)
    # Put window in center of screen
    root.eval('tk::PlaceWindow . center')
    root.mainloop()